# read

> PDFs, scans and page images read page by page: a sound text layer kept, scanned or garbled pages read by a vision model, every page's route and versions kept

In [ ]:
#| default_exp read

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *
import tempfile

`read_doc` reads one file without a vault and returns its pages; `Vault.read_doc` reads a file or a folder into the vault and keeps every page's text, route and flags as versions. `parse` picks the route: `auto` keeps a text layer that passes the checks, OCRs scanned pages, and sends what is still scanned or flagged to `model`; `text` keeps the text layer alone; `ocr` OCRs every page; `llm` sends every page to `model`. `model` is anything `urai` takes, local or hosted; None reads `$VISHALAKSHI_MODEL`.

In [ ]:
#| export
import json, re, time
from concurrent.futures import ThreadPoolExecutor
from fastcore.all import AttrDict, L, Path, first, ifnone, patch
from pdflite import pdf_doc, pdf_pages
from urai.record import resp_text
from litesearch import write_txn
from vishalakshi.core import Vault

## Prompts

`PRESETS` are reading prompts for a vision model, each asking for JSON items in reading order. `generic` suits any printed page; `sanskrit` and `tamil` carry what those prints get wrong; `tables` reads tables as rows. `reading_prompt` fills one in with the document's `title`, `script` and a `hint` (how this print numbers its verses, what to skip). A preset that is not a name is taken as a whole prompt.

In [ ]:
#| export
PRESETS = dict(
    generic=("You transcribe a scanned page{of}. Return JSON only, no prose, no code fence: "
             '{{"items": [{{"kind": "heading"|"prose"|"verse"|"table"|"note", "n": <a number printed with it, or null>, "text": "<exactly as printed>", '
             '"rows": [["cell", …], …]}}]}} in reading order. A heading is a title set on its own line. Prose: one item per paragraph, its lines joined '
             "with a space, a word hyphenated at a line end rejoined. A verse keeps its printed lines joined by \\n. A table gives `rows`, one list of cells "
             "per printed row, and an empty `text`. Skip running heads, page numbers and library stamps. Keep the spelling and script as printed and never "
             "rewrite a clear reading. {hint} If the page has no text, return {{\"items\": []}}."),
    sanskrit=("You transcribe scanned pages of a printed {script} text{of}. Return JSON only, no prose, no code fence: "
              '{{"items": [{{"kind": "verse"|"prose"|"heading"|"colophon", "n": <the verse number printed at its end, or null>, "text": "<exactly as printed>"}}]}} '
              "in reading order. Transcribe ONLY the original-language text: verses, prose, a chapter heading in that language (e.g. अथ … अध्यायः), and a "
              "chapter-end colophon (इति …). Ignore English translation, commentary, notes, running heads and page numbers. For a verse, join its printed "
              "lines with \\n and keep the closing ॥ n ॥. A verse that runs off the bottom of the page: give its lines with n null. Read every akṣara "
              "carefully: this kind of typeface makes ष look like प, ी like ो, द्व like ह, and conjuncts like म्र, द्य, ष्ण, र्द्ध easy to drop. Where the print "
              "is ambiguous choose the reading that is a real word and scans, but never rewrite a clear reading. Keep sandhi, avagraha and daṇḍas as "
              "printed. {hint} If the page has no such text, return {{\"items\": []}}."),
    tamil=("You transcribe scanned pages of a printed Tamil book{of}. Return JSON only, no prose, no code fence: "
           '{{"items": [{{"kind": "verse"|"prose"|"heading"|"colophon", "n": <a number printed with it, as an integer, or null>, "text": "<exactly as printed>"}}]}} '
           "in reading order. A verse keeps its printed lines joined by \\n; prose is one item per paragraph, its lines joined with a space. Write old "
           "letter forms in modern Unicode (the old ணா, றா, லை, னை glyphs as ணா, றா, லை, னை) and keep the spelling as printed. Old print confuses ண/ன, "
           "ள/ழ/ல, ற/ர and drops the puḷḷi: read each word as the Tamil it must be, but never rewrite a clear reading. Skip running heads and page "
           "numbers. {hint} If the page has no such text, return {{\"items\": []}}."),
    tables=("You transcribe the tables on a scanned page{of}. Return JSON only, no prose, no code fence: "
            '{{"items": [{{"kind": "table", "title": <the heading over it or null>, "rows": [["cell", …], …]}} | {{"kind": "prose", "text": …}}]}} '
            "in reading order, header row first, each cell exactly as printed. Text that is not a table is prose. {hint} If the page has no text, "
            "return {{\"items\": []}}."))

def reading_prompt(name='generic', title='', script='', hint='') -> str:
    "The reading prompt `name`, or `name` itself when it is a whole prompt, filled in."
    return PRESETS.get(name, name).format(of=f': {title}' if title else '', script=script or 'Devanagari', hint=hint)

In [ ]:
assert 'Gita' in reading_prompt('sanskrit', 'Gita') and 'ष look like प' in reading_prompt('sanskrit')
assert 'Tamil' in reading_prompt('tamil')
test_eq(reading_prompt('Read it: {hint}', hint='x'), 'Read it: x')

## Pages as images

`render` draws a page with pdf-oxide: PNG at `dpi`, or a JPEG within `max_px` when the PNG would be over `max_bytes`, which some hosted models refuse.

In [ ]:
#| export
def render(pdf, page:int, dpi:int=200, max_px:int=3000, max_bytes:int=3_000_000) -> bytes:
    "Page `page` (0-based) as PNG, or as JPEG within `max_px` when the PNG is over `max_bytes`."
    d = pdf_doc(pdf)
    b = d.render_page(page, dpi=dpi, format='png')
    return b if len(b) <= max_bytes else d.render_page_fit(page, max_px, max_px, format='jpeg')

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.webp')

In [ ]:
from pdf_oxide import Pdf
_fx = Path(tempfile.mkdtemp())
for _i, _t in enumerate(['# Attention Is All You Need\n\nThe dominant sequence transduction models are based on recurrent or convolutional networks.',
                         '# Background\n\nThe goal of reducing sequential computation also forms the foundation of the Extended Neural GPU.']):
    Pdf.from_markdown(_t).save(str(_fx/f'p{_i}.pdf'))
Pdf.merge([str(_fx/'p0.pdf'), str(_fx/'p1.pdf')]).save(str(_fx/'two.pdf'))
_src = _fx/'two.pdf'
_png = render(_src, 0, dpi=72)
assert _png[:4] == b'\x89PNG'
assert render(_src, 0, dpi=300, max_bytes=1000)[:2] == b'\xff\xd8'   # over the limit: a JPEG

## Checks

`flags` names what looks wrong with a page's text: `thin` (under `min_chars` letters), `replacement` (U+FFFD), `legacy` (Latin-1 and typographic characters a legacy Devanagari font's text layer reads as), `split` (a line opening on a dependent vowel sign or virāma), `akshara` (a malformed akṣara: a virāma before a vowel sign, two vowel signs, and the like), `foreign` (Greek or Cyrillic beside Devanagari). `disagree` is the letter edit ratio of two readings, vowel signs kept, daṇḍas, digits and punctuation dropped: 0.1 or more is worth a look.

In [ ]:
#| export
DEP = re.compile('(?m)^[ऀ-ःऺ-़ा-्ॎॏ॑-ॗॢॣா-்ௗ]')                    # a line opening on a dependent sign: a split akṣara
AKSHARA = re.compile('्[ा-ौॢॣ]|[ा-ॄॆ-ौ][ा-ॄॆ-ौ]|(?<!ग)्[ंः]|[ा-्]़|[ऄ-औ][ा-ौ]')   # malformed Devanagari akṣaras
FOREIGN = re.compile('[ऀ-ॿ][Ͱ-ϿЀ-ӿ]|[Ͱ-ϿЀ-ӿ][ऀ-ॿ]')
LEGACY = re.compile('[¡-ÿ‘-„†-‰]')       # what a legacy Devanagari font's text layer reads as
def flags(text:str, min_chars:int=20) -> list:
    "What looks wrong with a page's text: `thin`, `replacement`, `legacy`, `split`, `akshara`, `foreign`."
    t = re.sub(r'!\[[^\]]*\]\([^)]*\)', '', text or '')
    letters = re.sub(r'[\W\d_]', '', t)
    out = []
    if len(letters) < min_chars: out.append('thin')
    if '�' in t: out.append('replacement')
    if letters and len(LEGACY.findall(t)) > .05 * len(letters): out.append('legacy')
    if DEP.search(t): out.append('split')
    if AKSHARA.search(t): out.append('akshara')
    if FOREIGN.search(t): out.append('foreign')
    return out

_LETTERS = re.compile(r'[\s\d०-९।॥|ऽ\u200c\u200d!-/:-@\[-`{-~–—‘’“”]')   # spaces, digits, daṇḍas, avagraha, joiners, punctuation; vowel signs stay
def disagree(a:str, b:str) -> float|None:
    "Letter edits between two readings over the longer's letters; None when neither has any."
    a, b = _LETTERS.sub('', a or ''), _LETTERS.sub('', b or '')
    if not (a or b): return None
    p = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        c = [i]
        for j, y in enumerate(b, 1): c.append(min(p[j] + 1, c[j-1] + 1, p[j-1] + (x != y)))
        p = c
    return p[-1] / max(len(a), len(b))

In [ ]:
test_eq(flags('Hello world this is enough letters'), [])
test_eq(flags('क्ा and more letters here to pass'), ['akshara'])
assert 'split' in flags('कृष्ण\nा is a sign broken off here, long enough')
assert 'legacy' in flags('Jke ‘kfDr ¼ji¼ Hkxoku ¾ ™ more text here')
test_eq(flags(''), ['thin'])
assert disagree('रामः', 'रामा') > 0 and disagree('राम ॥१॥', 'राम') == 0
test_eq(disagree('', ''), None)

## Reading a page

`read_page` sends one page image to a vision model through `CachedChat`, so a reading is recorded once and replayed after. It returns the blocks, their markdown, the model and the cost. `chat` takes any callable in its place: the tests pass a stand-in.

In [ ]:
#| export
def _md(items) -> str:
    "A page's items as markdown."
    def one(it):
        if it.get('rows'): return '\n'.join('| ' + ' | '.join(map(str, r)) + ' |' + ('\n|' + '---|' * len(r) if not i else '') for i, r in enumerate(it['rows']))
        t = (it.get('text') or '').strip()
        return f'## {t}' if it.get('kind') == 'heading' and t else t
    return '\n\n'.join(s for s in map(one, items or []) if s)

def _json(s):
    s = re.sub(r'^```(?:json)?\s*|\s*```$', '', (s or '').strip())
    return json.loads(s[s.find('{'):s.rfind('}') + 1])

def backends():
    "Register rishi's model backends with urai, which knows none of its own: `claude/sonnet` resolves only after."
    import rishi

def read_page(img:bytes, model:str=None, preset:str='generic', title:str='', script:str='', hint:str='', chat=None) -> AttrDict:
    "One page image read by a vision model: its `blocks`, their markdown `text`, the `model` and its `cost`."
    from vishalakshi.ask import CachedChat
    if chat is None: backends()
    c = chat or CachedChat(model, sp=reading_prompt(preset, title, script, hint), max_output_tokens=16000)
    try: r = c([img, 'Transcribe the text on this page.'])
    finally:
        if chat is None and hasattr(c, 'close'): c.close()
    items = _json(resp_text(r) if not isinstance(r, str) else r).get('items') or []
    return AttrDict(blocks=items, text=_md(items), model=getattr(c, 'model', model), cost=((r.get('usage') or {}).get('cost') if isinstance(r, dict) else None))

In [ ]:
class Fake:
    "A vision model stand-in: it answers every page with the same two items, in a code fence."
    model, calls = 'fake/vision', 0
    def __call__(self, msg):
        Fake.calls += 1
        return dict(content='```json\n{"items":[{"kind":"heading","text":"Attention Is All You Need"},{"kind":"prose","text":"The dominant sequence transduction models are based on recurrent networks."}]}\n```', usage=dict(cost=0.01))
_r = read_page(_png, chat=Fake())
test_eq([b['kind'] for b in _r.blocks], ['heading', 'prose'])
assert _r.text.startswith('## Attention') and _r.cost == .01
test_eq(_md([dict(kind='table', rows=[['a', 'b'], ['1', '2']])]), '| a | b |\n|---|---|\n| 1 | 2 |')

In [ ]:
backends()
from urai.opts import resolve_runtime
test_eq(resolve_runtime('claude/sonnet'), ('claude', 'sonnet'))

## Reading a document

`read_doc` takes a PDF or a page image. Each page comes back as `{page, text, route, flags}`, with `model` and `cost` on a page the model read; `route` is `text`, `ocr`, `scanned` (nothing read it) or `model`. `pages` limits what goes to the model. `doc_files` lists the PDFs and images under a folder.

In [ ]:
#| export
PARSE = ('auto', 'text', 'ocr', 'llm')
def as_list(x, f=str):
    "`x` as a list of `f`: a comma-separated string, as a command line gives it, split; None stays None."
    return None if x is None else [f(v) for v in (x.split(',') if isinstance(x, str) else x) if str(v).strip()]
def read_doc(path, parse:str='auto', model:str=None, preset:str='generic', title:str='', script:str='', hint:str='',
             pages=None, dpi:int=200, out=None, workers:int=4, chat=None) -> L:
    """Each page of a PDF or image as `{page, text, route, flags}`, with `blocks`, `model` and `cost` where the model read it. `parse`: `auto` keeps a sound text layer and sends scanned or flagged pages to
    `model`; `text` the text layer only; `ocr` OCR every page; `llm` the model on every page."""
    assert parse in PARSE, f'parse is one of {PARSE}'
    pages = as_list(pages, int)
    p = Path(path)
    if p.suffix.lower() in IMG_EXTS: rows = [dict(page=0, text='', route='scanned')]
    else: rows = pdf_pages(p, out or p.parent/'.read', ocr=dict(auto='auto', text='off', ocr='on', llm='off')[parse])
    rows = L(AttrDict(r, flags=flags(r['text'])) for r in rows)
    want = pages if pages is None else set(pages)
    def needs(r): return parse == 'llm' or (parse == 'auto' and (r.route == 'scanned' or bool(r.flags)))
    todo = [r for r in rows if (want is None or r.page in want) and needs(r)]
    def go(r):
        img = p.read_bytes() if p.suffix.lower() in IMG_EXTS else render(p, r.page, dpi)
        got = read_page(img, model, preset, title, script, hint, chat=chat)
        if got.text.strip(): r.update(text=got.text, route='model', model=got.model, cost=got.cost, flags=flags(got.text), blocks=got.blocks)
        return r
    if todo:
        with ThreadPoolExecutor(max(1, workers)) as ex: list(ex.map(go, todo))
    return rows

def doc_files(folder, exts=('.pdf',) + IMG_EXTS) -> L:
    "The PDFs and page images under `folder`, sorted."
    return L(sorted(f for f in Path(folder).rglob('*') if f.is_file() and f.suffix.lower() in exts and not any(s.startswith('.') for s in f.parts)))

In [ ]:
_d = Path(tempfile.mkdtemp())
_scan = _d/'scan.pdf'; _scan.write_bytes(pdf_doc(pdf_doc(_src).extract_pages_to_bytes([0])).flatten_to_images(dpi=150))
Fake.calls = 0
_t = read_doc(_src, chat=Fake(), out=_d/'.out')
test_eq(_t.attrgot('route'), ['text', 'text']); test_eq(Fake.calls, 0)          # a sound text layer goes to no model
_s = read_doc(_scan, parse='text', out=_d/'.out'); test_eq(_s[0].route, 'scanned')
_m = read_doc(_scan, parse='llm', chat=Fake(), out=_d/'.out')
test_eq(_m[0].route, 'model'); test_eq(_m[0].model, 'fake/vision')
(_d/'pg.png').write_bytes(_png)
test_eq(read_doc(_d/'pg.png', chat=Fake())[0].route, 'model')
test_eq(doc_files(_d).map(lambda f: f.name), ['pg.png', 'scan.pdf'])

## Into the vault

`Vault.read_doc` reads a file or every file under a folder, indexes it as one document, and keeps each page in `read_pages`: a new version only when a page's text or route changed. `Vault.pages` gives a document's latest pages, or every version. `Vault.reparse` reads pages of a read document again, by default with the model, using the document's own preset, script and hint unless told otherwise, and re-indexes it.

In [ ]:
#| export
@patch
def _pages_t(self:Vault):
    t = self.db.t.read_pages
    if not t.exists():
        with write_txn(self.db):
            t.create(doc_id=str, store=str, page=int, version=int, route=str, text=str, flags=str, model=str, preset=str, at=float, blocks=str,
                     pk=('doc_id', 'store', 'page', 'version'))
    elif 'blocks' not in t.columns_dict: self.db.execute('alter table read_pages add column blocks text')
    return t

@patch
def pages(self:Vault, doc_id:str, history:bool=False) -> L:
    "A read document's pages, its latest version of each, or every version with `history`."
    rs = L(self._pages_t().rows_where('doc_id=? and store=?', [doc_id, self.name], order_by='page, version'))
    if history: return rs.map(AttrDict)
    last = {}
    for r in rs: last[r['page']] = r
    return L(AttrDict(r, flags=json.loads(r['flags'] or '[]'), blocks=json.loads(r.get('blocks') or 'null')) for r in last.values())

@patch
def _file_pages(self:Vault, doc_id, rows, preset):
    t, now, have = self._pages_t(), time.time(), {r.page: r for r in self.pages(doc_id)}
    with write_txn(self.db):
        for r in rows:
            if (o := have.get(r.page)) and o.text == r.text and o.route == r.route and not r.get('blocks'): continue
            t.insert(dict(doc_id=doc_id, store=self.name, page=r.page, version=(o.version if o else 0) + 1, route=r.route, text=r.text,
                          flags=json.dumps(r.flags), model=r.get('model'), preset=preset if r.route == 'model' else None, at=now,
                          blocks=json.dumps(r['blocks'], ensure_ascii=False) if r.get('blocks') else None))

@patch
def read_doc(self:Vault, src, title:str=None, kind:str=None, parse:str='auto', model:str=None, preset:str='generic', script:str='', hint:str='',
         pages=None, dpi:int=200, workers:int=4, chat=None, **kw):
    """Read a PDF, a page image, or every one under a folder into the vault, page by page: a sound text layer kept, scanned or flagged pages read
    by `model`. Each page's text, route and flags are kept as versions (`pages`). Returns the ingest result, with `routes` and `flagged`."""
    p = Path(src)
    if p.is_dir(): return doc_files(p).map(lambda f: self.read_doc(f, None, kind, parse, model, preset, script, hint, pages, dpi, workers, chat, **kw))
    rows = read_doc(p, parse, model, preset, title or '', script, hint, pages, dpi, self.assets(p.stem), workers, chat)
    out = self.add([(r.page, r.text) for r in rows], title or p.stem, source=str(p), kind=kind or ('pdf' if p.suffix.lower() == '.pdf' else 'image'),
                   meta=dict(parse=parse, model=model, preset=preset, script=script, hint=hint), force=True, **kw)
    self._file_pages(out['doc_id'], rows, preset)
    return dict(out, routes={k: len(v) for k, v in rows.groupby(lambda r: r.route).items()},
                flagged=[r.page for r in rows if r.flags])

@patch
def reparse(self:Vault, doc_id:str, pages=None, model:str=None, preset:str=None, script:str=None, hint:str=None, parse:str='llm',
            dpi:int=200, workers:int=4, chat=None) -> dict:
    "Read `pages` of a read document again (all when None), by `model` with `parse='llm'`; the new texts become new versions and the document is re-indexed."
    d, pages = first(self.t.docs(where='id=?', where_args=[doc_id])), as_list(pages, int)
    assert d, f'no document {doc_id}'
    m = json.loads(d['meta'] or '{}')
    preset, script, hint = ifnone(preset, m.get('preset', 'generic')), ifnone(script, m.get('script', '')), ifnone(hint, m.get('hint', ''))
    old = {r.page: r for r in self.pages(doc_id)}
    got = read_doc(d['source'], parse, model, preset, d['title'], script, hint, pages, dpi, self.assets(Path(d['source']).stem), workers, chat)
    keep = got if pages is None else got.filter(lambda r: r.page in set(pages))
    self._file_pages(doc_id, keep, preset)
    new = {r.page: r for r in self.pages(doc_id)}
    out = self.add([(k, new[k].text) for k in sorted(new)], d['title'], source=d['source'], kind=d['kind'], meta=dict(m, reparsed=time.time()), force=True)
    return dict(out, pages=sorted(r.page for r in keep), changed=[k for k in new if old.get(k) is None or old[k].text != new[k].text])

In [ ]:
_v = Vault(Path(tempfile.mkdtemp())/'v.db', offline=True)
_o = _v.read_doc(_scan, parse='text')
test_eq(_o['routes'], {'scanned': 1}); test_eq(_o['flagged'], [0])
test_eq([(p.page, p.version, p.route) for p in _v.pages(_o['doc_id'])], [(0, 1, 'scanned')])
_r = _v.reparse(_o['doc_id'], chat=Fake())
test_eq(_r['changed'], [0]); test_eq([p.version for p in _v.pages(_o['doc_id'], history=True)], [1, 2])
test_eq(_v.pages(_o['doc_id'])[0].route, 'model')
test_eq([b['kind'] for b in _v.pages(_o['doc_id'])[0].blocks], ['heading', 'prose'])
assert 'dominant sequence' in _v.search('dominant sequence transduction')[0]['snippet']
_v.read_doc(_scan, parse='text'); test_eq(len(_v.pages(_o['doc_id'], history=True)), 3)   # read again as text: the text route is a new version
test_eq(len(_v.read_doc(_d, parse='text')), 2)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()